# Seeding procedure

The following notebook loads the [arXiv Scientific Research Papers Dataset](https://www.kaggle.com/datasets/sumitm004/arxiv-scientific-research-papers-dataset), embeds it (fully or partially) and saves those embeddings in a ChromaDB instance using `"type": "seed"` within its metadata.

In [ ]:
import polars as pl

from math import ceil
from tqdm.notebook import tqdm

from scifind.integrations.chroma import get_chroma_client, ALL_DOCUMENTS_COLLECTION_NAME
from scifind.integrations.dataset import load_arxiv_dataset
from scifind.integrations.model import get_embedding_model

### Configs

In [ ]:
# Fraction of the dataset that will be used to seed the database, useful when testing on a CPU
DATASET_SAMPLE_FRACTION: float | int | pl.Series = 0.001

# Batch size for embedding (passed to the SentenceTransformer)
EMBEDDING_BATCH_SIZE: int = 4

In [ ]:
chroma = get_chroma_client()
collection = chroma.get_or_create_collection(
    ALL_DOCUMENTS_COLLECTION_NAME, configuration={"hnsw": {"space": "cosine"}}
)

In [ ]:
model = get_embedding_model()

In [ ]:
dataset = (
    load_arxiv_dataset()
    .select("id", "title", "category_code", "authors", "summary")
    .collect()
    .sample(fraction=DATASET_SAMPLE_FRACTION)
)

print(f"Dataset shape: {dataset.shape}")

dataset.head()

In [ ]:
max_chroma = chroma.get_max_batch_size()
batch_size = max(
    EMBEDDING_BATCH_SIZE, (max_chroma // EMBEDDING_BATCH_SIZE) * EMBEDDING_BATCH_SIZE
)
num_batches = ceil(len(dataset) / batch_size)

print(
    f"Embeddings will be done in storage batches of {batch_size} vectors according to ChromaDB's max limits ({max_chroma}) and the set EMBEDDING_BATCH_SIZE ({EMBEDDING_BATCH_SIZE})"
)

batches = dataset.iter_slices(batch_size)

In [ ]:
for batch in tqdm(batches, desc="Committing embeddings to Chroma", total=num_batches):
    summaries = batch["summary"].to_list()
    embeddings = model.encode(
        summaries, batch_size=EMBEDDING_BATCH_SIZE, show_progress_bar=True
    )

    metadatas = batch.select(
        type=pl.lit("seed"),
        title=pl.col("title").fill_null(""),
        category_code=pl.col("category_code").fill_null(""),
        authors=(
            pl.col("authors").list.join(", ")
            if batch["authors"].dtype == pl.List
            else pl.col("authors").fill_null("")
        ),
    ).to_dicts()

    collection.add(
        ids=batch["id"].to_list(),
        embeddings=embeddings,
        documents=summaries,
        metadatas=metadatas,
    )